# AutoML-Setup: Configuración de Entornos

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 Objetivo

Crear los entornos conda aislados para cada framework AutoML.
Se ejecuta **una sola vez** antes de usar los notebooks M02-M05.

## ❓ ¿Por qué entornos separados?

Cada framework AutoML tiene dependencias que chocan entre sí (versiones incompatibles de 
scikit-learn, numpy, etc.). Aislarlos en entornos conda evita conflictos y mantiene 
`tfm_abandono` limpio para el resto del TFM (Fases 4-7).

## 📋 Entornos

| Entorno | Framework | Notebook | Kernel Jupyter |
|---------|-----------|----------|----------------|
| `tfm_abandono` | sklearn (baselines) | M00, M01, M06 | Python (TFM Abandono) — ya existe |
| `env_lazypredict` | LazyPredict | M02 | Python (LazyPredict) |
| `env_pycaret` | PyCaret 3.x | M03 | Python (PyCaret) |
| `env_h2o` | H2O AutoML | M04 | Python (H2O) |
| `env_autogluon` | AutoGluon | M05 | Python (AutoGluon) |

## ⚠️ Requisitos previos

- Anaconda o Miniconda instalado
- Conexión a internet
- ~5-8 GB de espacio en disco
- Java 8+ (solo para H2O)

## 🚀 Cómo usar después

No hace falta cambiar el kernel a mano. Cada notebook ya tiene configurado su kernel 
correcto en los metadatos. Al abrirlo, Jupyter lo selecciona automáticamente.

In [ ]:
# ============================================================================
# CELDA 1: VERIFICACIÓN DEL SISTEMA
# ============================================================================

import sys, os, subprocess, platform

print('=' * 60)
print('VERIFICACIÓN DEL SISTEMA')
print('=' * 60)

so = platform.system()
print(f'\n✓ Sistema operativo: {so}')
print(f'✓ Python: {sys.version.split()[0]}')
print(f'✓ Arquitectura: {platform.machine()}')

try:
    result = subprocess.run(['conda', '--version'], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print(f'✓ Conda: {result.stdout.strip()}')
        CONDA_OK = True
    else:
        print('✗ Conda: error al ejecutar')
        CONDA_OK = False
except FileNotFoundError:
    print('✗ Conda: NO encontrado')
    print('  → Instalar: https://docs.conda.io/en/latest/miniconda.html')
    CONDA_OK = False
except Exception as e:
    print(f'✗ Conda: {e}')
    CONDA_OK = False

# Java (para H2O)
result = subprocess.run('java -version', shell=True, capture_output=True, text=True, timeout=10)
if result.returncode == 0:
    print(f'✓ Java: {result.stderr.split(chr(10))[0]}')
else:
    print('⚠️ Java no encontrado (necesario para H2O)')
    print('  → Instalar: https://adoptium.net/')

if CONDA_OK:
    print('\n✅ Sistema listo')
else:
    print('\n❌ Instala conda antes de continuar')

In [ ]:
# ============================================================================
# CELDA 2: VERIFICAR ENTORNOS EXISTENTES
# ============================================================================

print('=' * 60)
print('ENTORNOS CONDA EXISTENTES')
print('=' * 60)

ENTORNOS_NECESARIOS = {
    'env_lazypredict': 'LazyPredict',
    'env_pycaret': 'PyCaret 3.x',
    'env_h2o': 'H2O AutoML',
    'env_autogluon': 'AutoGluon'
}

try:
    result = subprocess.run(['conda', 'env', 'list'], capture_output=True, text=True, timeout=30)
    envs_texto = result.stdout
    print(envs_texto)
    
    entornos_existentes = []
    for line in envs_texto.split('\n'):
        if line.strip() and not line.startswith('#'):
            entornos_existentes.append(line.split()[0])
    
    print('-' * 60)
    faltan = 0
    for env, desc in ENTORNOS_NECESARIOS.items():
        if env in entornos_existentes:
            print(f'  ✅ {env} ({desc}) - YA EXISTE')
        else:
            print(f'  ❌ {env} ({desc}) - FALTA')
            faltan += 1
    
    if faltan == 0:
        print('\n✅ Todos los entornos existen. Puedes saltar a la verificación final (celda 7).')
    else:
        print(f'\n⚠️ Faltan {faltan} entornos. Ejecuta las celdas correspondientes.')
except Exception as e:
    print(f'Error: {e}')

## 🔧 Crear entornos

Ejecuta **solo las celdas de los entornos que falten**.
Cada celda tarda 3-15 minutos según tu conexión.

In [ ]:
# ============================================================================
# CELDA 3: CREAR ENTORNO LazyPredict
# ============================================================================
# ⏱️ Tiempo estimado: 3-5 minutos
# 📦 Tamaño: ~500 MB
# ============================================================================

print('=' * 60)
print('CREANDO ENTORNO: env_lazypredict')
print('=' * 60)

comandos = [
    'conda create -n env_lazypredict python=3.11 -y',
    'conda run -n env_lazypredict pip install lazypredict scikit-learn pandas pyarrow jupyter ipykernel matplotlib --quiet',
    'conda run -n env_lazypredict python -m ipykernel install --user --name env_lazypredict --display-name "Python (LazyPredict)"',
]

for i, cmd in enumerate(comandos, 1):
    print(f'\n[{i}/{len(comandos)}] {cmd[:70]}...')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print(f'  ⚠️ {result.stderr[:200]}')
    else:
        print(f'  ✓ OK')

result = subprocess.run('conda run -n env_lazypredict python -c "import lazypredict; print(lazypredict.__version__)"',
                        shell=True, capture_output=True, text=True, timeout=30)
if result.returncode == 0:
    print(f'\n✅ LazyPredict {result.stdout.strip()} instalado')
else:
    print(f'\n❌ Error: {result.stderr[:200]}')

In [ ]:
# ============================================================================
# CELDA 4: CREAR ENTORNO PyCaret
# ============================================================================
# ⏱️ Tiempo estimado: 5-10 minutos
# 📦 Tamaño: ~2 GB
# ============================================================================

print('=' * 60)
print('CREANDO ENTORNO: env_pycaret')
print('=' * 60)

comandos = [
    'conda create -n env_pycaret python=3.11 -y',
    'conda run -n env_pycaret pip install pycaret[full]>=3.0 jupyter ipykernel pandas pyarrow --quiet',
    'conda run -n env_pycaret python -m ipykernel install --user --name env_pycaret --display-name "Python (PyCaret)"',
]

for i, cmd in enumerate(comandos, 1):
    print(f'\n[{i}/{len(comandos)}] {cmd[:70]}...')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print(f'  ⚠️ {result.stderr[:200]}')
    else:
        print(f'  ✓ OK')

result = subprocess.run('conda run -n env_pycaret python -c "import pycaret; print(pycaret.__version__)"',
                        shell=True, capture_output=True, text=True, timeout=30)
if result.returncode == 0:
    print(f'\n✅ PyCaret {result.stdout.strip()} instalado')
else:
    print(f'\n❌ Error: {result.stderr[:200]}')

In [ ]:
# ============================================================================
# CELDA 5: CREAR ENTORNO H2O
# ============================================================================
# ⏱️ Tiempo estimado: 3-5 minutos
# 📦 Tamaño: ~1 GB
# ⚠️ Requiere Java 8+
# ============================================================================

print('=' * 60)
print('CREANDO ENTORNO: env_h2o')
print('=' * 60)

comandos = [
    'conda create -n env_h2o python=3.11 -y',
    'conda run -n env_h2o pip install h2o>=3.44 jupyter ipykernel pandas pyarrow requests --quiet',
    'conda run -n env_h2o python -m ipykernel install --user --name env_h2o --display-name "Python (H2O)"',
]

for i, cmd in enumerate(comandos, 1):
    print(f'\n[{i}/{len(comandos)}] {cmd[:70]}...')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print(f'  ⚠️ {result.stderr[:200]}')
    else:
        print(f'  ✓ OK')

result = subprocess.run('conda run -n env_h2o python -c "import h2o; print(h2o.__version__)"',
                        shell=True, capture_output=True, text=True, timeout=30)
if result.returncode == 0:
    print(f'\n✅ H2O {result.stdout.strip()} instalado')
else:
    print(f'\n❌ Error: {result.stderr[:200]}')

# Verificar Java
result = subprocess.run('java -version', shell=True, capture_output=True, text=True, timeout=10)
if result.returncode == 0:
    print(f'✅ Java: {result.stderr.split(chr(10))[0]}')
else:
    print('⚠️ Java no encontrado. H2O lo requiere.')
    print('  → https://adoptium.net/')

In [ ]:
# ============================================================================
# CELDA 6: CREAR ENTORNO AutoGluon
# ============================================================================
# ⏱️ Tiempo estimado: 5-15 minutos (el más pesado)
# 📦 Tamaño: ~3 GB
# ============================================================================

print('=' * 60)
print('CREANDO ENTORNO: env_autogluon')
print('=' * 60)

comandos = [
    'conda create -n env_autogluon python=3.11 -y',
    'conda run -n env_autogluon pip install autogluon jupyter ipykernel pandas pyarrow --quiet',
    'conda run -n env_autogluon python -m ipykernel install --user --name env_autogluon --display-name "Python (AutoGluon)"',
]

for i, cmd in enumerate(comandos, 1):
    print(f'\n[{i}/{len(comandos)}] {cmd[:70]}...')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=900)
    if result.returncode != 0:
        print(f'  ⚠️ {result.stderr[:200]}')
    else:
        print(f'  ✓ OK')

result = subprocess.run('conda run -n env_autogluon python -c "import autogluon; print(autogluon.__version__)"',
                        shell=True, capture_output=True, text=True, timeout=30)
if result.returncode == 0:
    print(f'\n✅ AutoGluon {result.stdout.strip()} instalado')
else:
    print(f'\n❌ Error: {result.stderr[:200]}')

In [ ]:
# ============================================================================
# CELDA 7: VERIFICACIÓN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('VERIFICACIÓN FINAL DE TODOS LOS ENTORNOS')
print('=' * 60)

entornos = [
    ('tfm_abandono', 'sklearn', 'Baselines sklearn', 'M00, M01, M06'),
    ('env_lazypredict', 'lazypredict', 'LazyPredict', 'M02'),
    ('env_pycaret', 'pycaret', 'PyCaret', 'M03'),
    ('env_h2o', 'h2o', 'H2O AutoML', 'M04'),
    ('env_autogluon', 'autogluon', 'AutoGluon', 'M05'),
]

todos_ok = True
for env, lib, desc, nbs in entornos:
    result = subprocess.run(
        f'conda run -n {env} python -c "import {lib}; print({lib}.__version__)"',
        shell=True, capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0:
        v = result.stdout.strip()
        print(f'  ✅ {env:20s} | {desc:15s} v{v:10s} | {nbs}')
    else:
        print(f'  ❌ {env:20s} | {desc:15s} FALTA         | {nbs}')
        todos_ok = False

print()
if todos_ok:
    print('✅ TODOS LOS ENTORNOS LISTOS')
    print('\n📌 Siguiente: ejecutar notebooks AutoML en orden M00 → M06')
    print('   Cada notebook ya tiene configurado su kernel correcto.')
else:
    print('❌ Algunos entornos faltan. Ejecuta las celdas correspondientes.')
print('=' * 60)